<a href="https://colab.research.google.com/github/islayshi/CMPE295A-Project/blob/main/GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

In [ ]:
df = pd.read_csv('/content/sample_data/mnist_test.csv')
print(df.head())

   7  0  0.1  0.2  0.3  0.4  0.5  0.6  0.7  0.8  ...  0.658  0.659  0.660  \
0  2  0    0    0    0    0    0    0    0    0  ...      0      0      0   
1  1  0    0    0    0    0    0    0    0    0  ...      0      0      0   
2  0  0    0    0    0    0    0    0    0    0  ...      0      0      0   
3  4  0    0    0    0    0    0    0    0    0  ...      0      0      0   
4  1  0    0    0    0    0    0    0    0    0  ...      0      0      0   

   0.661  0.662  0.663  0.664  0.665  0.666  0.667  
0      0      0      0      0      0      0      0  
1      0      0      0      0      0      0      0  
2      0      0      0      0      0      0      0  
3      0      0      0      0      0      0      0  
4      0      0      0      0      0      0      0  

[5 rows x 785 columns]


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df.values)

In [ ]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)


time_step = 100
X, y = create_dataset(scaled_data, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

In [ ]:
model = Sequential()
model.add(GRU(units=50, return_sequences=True, input_shape=(X.shape[1], 1)))
model.add(GRU(units=50))
model.add(Dense(units=1))
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.fit(X, y, epochs=10, batch_size=32)

Epoch 1/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 41s 101ms/step - loss: 0.1088
Epoch 2/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 32s 102ms/step - loss: 0.1042
Epoch 3/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 31s 101ms/step - loss: 0.1033
Epoch 4/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 32s 102ms/step - loss: 0.1034
Epoch 5/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 30s 98ms/step - loss: 0.1032
Epoch 6/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 31s 99ms/step - loss: 0.1026
Epoch 7/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 42s 103ms/step - loss: 0.1021
Epoch 8/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 31s 98ms/step - loss: 0.1019
Epoch 9/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 31s 100ms/step - loss: 0.1010
Epoch 10/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 31s 101ms/step - loss: 0.1007


In [ ]:
input_sequence = scaled_data[-time_step:, 0].reshape(1, time_step, 1)
predicted_values = model.predict(input_sequence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 366ms/step


In [ ]:
temp_predicted_values = np.zeros((1, scaled_data.shape[1]))
temp_predicted_values[0, 0] = predicted_values[0, 0]
inversed_predicted_value = scaler.inverse_transform(temp_predicted_values)
print(f"The predicted temperature for the next day is: {inversed_predicted_value[0][0]:.2f}°C")

The predicted temperature for the next day is: 6.15°C
